In [ ]:
# clone
import subprocess
import os

def clone_github_repository():
  # クローンしたいGitHubリポジトリのURL
  repository_url = "https://github.com/tosakazu/smash_database.git"
  repository_name = repository_url.split('/')[-1].replace('.git', '')

  # Colabの仮想環境にリポジトリをクローン
  try:
      subprocess.run(['git', 'clone', repository_url], check=True)
      print(f"リポジトリ '{repository_name}' をクローンしました。")
  except subprocess.CalledProcessError as e:
      print(f"エラーが発生しました: {e}")
      exit()

  # クローンしたリポジトリのパス
  repository_path = os.path.join(os.getcwd(), repository_name)
  print(f"クローンしたリポジトリのパス: {repository_path}")
  return repository_path

# 必要に応じて、クローンしたリポジトリを削除 (Colabの環境をクリーンアップする場合)
import shutil

def clean_up_repository(repository_path):
  shutil.rmtree(repository_path)
  print(f"\nクローンしたリポジトリ '{repository_path}' を削除しました。")

In [ ]:
import glob

# 検索したいファイル名のパターン (glob形式)
# file_pattern = "data/startgg/*.jsonl"
# file_pattern = "data/startgg/events/**/matches.json"
# file_pattern = "data/startgg/events/Japan/2025/04/*/*/*/matches.json"
# file_pattern = "data/startgg/events/*/*/*/*/*/*/matches.json"
# file_pattern = "*.txt"  # 例: 拡張子が .txt のファイル
# file_pattern = "data/*.csv" # 例: dataディレクトリ以下の .csv ファイル

def glob_repository_files(repository_path, file_pattern):
  # globを使ってファイルを検索
  search_path = os.path.join(repository_path, file_pattern) # サブディレクトリも含む
  print(f"検索するパス: {search_path}")
  # search_path = os.path.join(repository_path, "**", file_pattern) # サブディレクトリも含む
  found_files = glob.glob(search_path, recursive=True)
  return found_files

  # if found_files:
  #     print(f"\nパターン '{file_pattern}' に一致するファイル:")
  #     for file_path in found_files:
  #         print(file_path)
  # else:
  #     print(f"\nパターン '{file_pattern}' に一致するファイルは見つかりませんでした。")

#

In [ ]:
import datetime
import time
import pytz

def get_midnight_jst_unixtime(date_obj):
    """
    指定された日付（datetime.dateオブジェクト）の日本時間0時のUnixタイムスタンプを取得します。

    Args:
        date_obj: datetime.dateオブジェクト

    Returns:
        int: 日本時間0時のUnixタイムスタンプ（秒）
    """
    # 指定された日付の日本時間0時のdatetimeオブジェクトを作成
    jst = pytz.timezone('Asia/Tokyo')
    midnight_jst = datetime.datetime(date_obj.year, date_obj.month, date_obj.day, 0, 0, 0, tzinfo=jst)

    # UTCに変換
    midnight_utc = midnight_jst.astimezone(pytz.utc)

    # Unixタイムスタンプ（秒）を取得
    unixtime = int(midnight_utc.timestamp())
    return unixtime

def get_midnight_jst_unixtime_from_str(date_str):
    """
    'YYYY-mm-dd'形式の文字列を引数として、その日付の日本時間0時のUnixタイムスタンプを取得します。

    Args:
        date_str (str): 'YYYY-mm-dd'形式の日付文字列

    Returns:
        int: 日本時間0時のUnixタイムスタンプ（秒）
        None: 無効な日付形式の場合
    """
    try:
        # 文字列をdatetime.dateオブジェクトに変換
        date_obj = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()
        return get_midnight_jst_unixtime(date_obj)
    except ValueError:
        print(f"エラー: '{date_str}' は 'YYYY-mm-dd' 形式ではありません。")
        return None

def get_today_timestamp():
  # 今日の日付（UTC）を取得
  today_utc = datetime.datetime.now(datetime.timezone.utc).date()

  # 今日の午前0時（UTC）のdatetimeオブジェクトを作成
  today_midnight_utc = datetime.datetime(today_utc.year, today_utc.month, today_utc.day, 0, 0, 0, tzinfo=datetime.timezone.utc)

  # Unixタイム（エポック秒）に変換
  today_midnight_unixtime = int(today_midnight_utc.timestamp())

  return today_midnight_unixtime

def get_yyyymmddhhmmss():
  return datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime('%Y%m%d%H%M%S')

In [ ]:
import json
import os

def load_tournament_data(file_path):
  data_list = []
  try:
    with open(file_path, 'r') as f:
      for line in f:
        try:
          json_object = json.loads(line.strip())
          data_list.append(json_object)
        except json.JSONDecodeError as e:
          print(f"JSONデコードエラー: {e} (行: '{line.strip()}')")
  except FileNotFoundError:
      print(f"エラー: ファイル '{file_path}' が見つかりません。")
  return data_list

def get_event_attr(event_datum):
  attr_file = os.path.join(repository_path, event_datum["path"], "attr.json")
  if not os.path.exists(attr_file):
    return { "timestamp": None, "numEntrants": None }
  with open(attr_file, 'r') as f:
    attr_data = json.load(f)
    timestamp = attr_data["timestamp"] if "timestamp" in attr_data else None
    numEntrants = attr_data["num_entrants"] if "num_entrants" in attr_data else None
    return { "timestamp": timestamp, "numEntrants": numEntrants }
  return { "timestamp": None, "numEntrants": None }

########### ここまで関数定義 ###################

#### リポジトリ更新

print(os.listdir(os.getcwd()))
old_repository_path = os.path.join(os.getcwd(), "smash_database")
if os.path.exists(old_repository_path):
  clean_up_repository(old_repository_path)
repository_path = clone_github_repository()

#### ファイル読み込んで変数化

tournament_jsonl_file_path = os.path.join(repository_path, "data/startgg/tournaments.jsonl")
tournament_data = load_tournament_data(tournament_jsonl_file_path)
print(len(tournament_data))

event_data = []

for tournament_datum in tournament_data:
  if not isinstance(tournament_datum, dict) or "events" not in tournament_datum:
    continue
  for event_datum in tournament_datum["events"]:
    if not isinstance(event_datum, dict) or "path" not in event_datum:
      continue
    match_file = os.path.join(repository_path, event_datum["path"], "matches.json")
    if not os.path.exists(match_file):
      continue
    with open(match_file, 'r') as f:
      match_data = json.load(f)
      event_datum["tournament_name"] = tournament_datum["name"]
      event_datum["matches"] = match_data
      event_attr = get_event_attr(event_datum)
      event_datum["timestamp"] = int(event_attr["timestamp"])
      event_datum["numEntrants"] = int(event_attr["numEntrants"])
      event_data.append(event_datum)

#### 選手ペア → 対戦履歴の逆引きインデックスを構築
from collections import defaultdict
match_lookup = defaultdict(list)
for event_datum in event_data:
  ts = event_datum["timestamp"]
  ne = event_datum["numEntrants"]
  if "matches" not in event_datum or not isinstance(event_datum["matches"], dict) or "data" not in event_datum["matches"]:
    continue
  for m in event_datum["matches"]["data"]:
    if not isinstance(m, dict):
      continue
    wid = m.get("winner_id")
    lid = m.get("loser_id")
    if wid is None or lid is None:
      continue
    key = (min(wid, lid), max(wid, lid))
    match_lookup[key].append({"timestamp": ts, "numEntrants": ne})
print(f"match_lookup 構築完了: {len(match_lookup)} ペア")


In [ ]:
##### match_point計算部分 #####

from functools import lru_cache
from google.colab import userdata

REF_DATE = userdata.get('REF_DATE')

def calc_match_point_from_timestamps(match_timestamps):
  ref_unixtime = get_midnight_jst_unixtime_from_str(REF_DATE)
  match_point = sum(map(lambda timestamp: 4 ** ((timestamp - ref_unixtime) / 31536000.0), match_timestamps))
  return match_point

def search_player_matches(target_user_id, opponent_user_id, least_num_entrants=0):
  key = (min(target_user_id, opponent_user_id), max(target_user_id, opponent_user_id))
  return [m for m in match_lookup.get(key, []) if m["numEntrants"] >= least_num_entrants]

@lru_cache(maxsize=None)
def calc_match_point(target_user_id, opponent_user_id, target_user_hidden_value, opponent_user_hidden_value, least_num_entrants=0):
  matches = search_player_matches(target_user_id, opponent_user_id, least_num_entrants)
  return calc_match_point_from_timestamps([m["timestamp"] for m in matches]) - abs(target_user_hidden_value - opponent_user_hidden_value)


In [ ]:
##### 順番入れ替え部分 #####

import math
from google.colab import userdata

FIXED_SEED_NUM = int(userdata.get('FIXED_SEED_NUM'))
CONDITIONAL_LEAST_NUM_ENTRANTS = int(userdata.get('CONDITIONAL_LEAST_NUM_ENTRANTS'))
APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM = int(userdata.get('APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM'))
SEARCH_BREADTH_MULTIPLIER = int(userdata.get('SEARCH_BREADTH_MULTIPLIER'))

def calc_lrv(x):
  if x <= 1:
    return 0
  return int(math.floor(math.log2(x-1))+math.ceil(math.log2(x*2/3)))

BREADTH_CONST = 0

# 探索幅の計算
def calc_breadth(index):
  return int((max(1, math.floor(math.log2(index)) - BREADTH_CONST)) * SEARCH_BREADTH_MULTIPLIER)

def calc_opponent_index(index):
  winner_lrv = math.ceil(math.log2(index + 1))
  return int(math.pow(2, winner_lrv) - index - 1)

TEMPORARY_INITIAL_MATCH_VALUE = -10000

# Wave 制約ヘルパー（wave_pattern, wave_cycle_length, allowed_waves_map は cell-7 で設定）
# Wave 設定がない場合のデフォルト値（制約なし）
wave_pattern = {}
wave_cycle_length = 1
allowed_waves_map = {}

def get_wave(index):
  if not wave_pattern:
    return ''
  pos = (index % wave_cycle_length) + 1
  return wave_pattern.get(pos, '')

def get_allowed_waves(player):
  disc = str(player.get('discriminator', ''))
  return allowed_waves_map.get(disc, [])

def is_wave_valid(player, position_index):
  allowed = get_allowed_waves(player)
  if not allowed:
    return True  # 制約なし
  return get_wave(position_index) in allowed

def _record_wave_violation(wave_violations, initial_data, adjusted_data, player_idx):
  current_pos = len(adjusted_data)
  player = initial_data[player_idx]
  player_name = player.get('player_name', player.get('gamer_tag', 'Unknown'))
  wave = get_wave(current_pos)
  allowed = get_allowed_waves(player)
  wave_violations.append({
    'phaseseed': current_pos + 1,
    'player_name': player_name,
    'wave': wave,
    'allowed_waves': allowed,
  })
  print(f'[警告] Wave制約撤廃: phaseseed={current_pos + 1}, player={player_name}, wave={wave}, allowed={allowed}')

# 一番上のplayer側から見て、空いている一番上のところがmatch_valueが最小の場合かどうか
# 許可Waveでない位置はスキップして比較する
def is_adjusted_seed(initial_data, adjusted_data, target_initial_index):
  target_user_id = initial_data[target_initial_index]['user_id']
  breadth = calc_breadth(target_initial_index)
  max_index = int(min(target_initial_index + breadth, len(initial_data)))

  adjusted_match_value = TEMPORARY_INITIAL_MATCH_VALUE
  match_log = []

  for current_index in range(len(adjusted_data), max_index):
    # 許可Waveでない位置は比較対象から除外
    if not is_wave_valid(initial_data[target_initial_index], current_index):
      continue
    opponent_index = calc_opponent_index(current_index)
    if opponent_index >= len(initial_data):
      continue
    # 対戦相手が未決定なら最小かどうかは決定しない
    if (opponent_index >= len(adjusted_data)):
      return None
    opponent_user_id = adjusted_data[opponent_index]['user_id']
    least_num_entrants = CONDITIONAL_LEAST_NUM_ENTRANTS if current_index <= APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM and opponent_index <= APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM else 0
    current_match_value = calc_match_point(target_user_id, opponent_user_id, initial_data[target_initial_index].get('hidden_value', 0), initial_data[opponent_index].get('hidden_value', 0), least_num_entrants)

    player_name_for_log = adjusted_data[opponent_index].get('player_name', adjusted_data[opponent_index].get('gamer_tag', 'Unknown'))
    match_log.extend([opponent_index, adjusted_data[opponent_index]['user_id'], player_name_for_log, current_match_value])
    if (adjusted_match_value <= TEMPORARY_INITIAL_MATCH_VALUE):
      adjusted_match_value = current_match_value
      continue
    if (current_match_value < adjusted_match_value):
      return None
  return match_log

def get_target_indices(initial_data, adjusted_data, ignore_wave=False):
  current_pos = len(adjusted_data)
  breadth = calc_breadth(current_pos)
  adjusted_user_ids = set(row['user_id'] for row in adjusted_data)
  max_index = int(min(len(initial_data), current_pos + breadth))
  indices = []
  for i in range(0, max_index):
    if initial_data[i]['user_id'] in adjusted_user_ids:
      continue
    if not ignore_wave and not is_wave_valid(initial_data[i], current_pos):
      continue
    indices.append(i)
  return indices

def get_tight_group(initial_data, adjusted_data):
  """
  w人の選手がcurrent_pos〜current_pos+w-1のw枠にしか置けない（タイトグループ）を検出する。
  単一選手の強制（w=1）を含む汎化版。見つかればその選手インデックスリストを返す。
  """
  current_pos = len(adjusted_data)
  placed_ids = set(row['user_id'] for row in adjusted_data)
  max_breadth = calc_breadth(current_pos)

  def valid_positions(player_idx):
    b = calc_breadth(player_idx)
    max_pos = min(player_idx + b, len(initial_data))
    return [j for j in range(current_pos, max_pos)
            if is_wave_valid(initial_data[player_idx], j)]

  unplaced = [i for i in range(len(initial_data))
              if initial_data[i]['user_id'] not in placed_ids]

  for w in range(1, max_breadth + 1):
    window_end = current_pos + w
    constrained = []
    for idx in unplaced:
      vp = valid_positions(idx)
      if not vp:
        continue
      if all(j < window_end for j in vp):
        constrained.append(idx)
    if len(constrained) >= w:
      return constrained

  return None

def get_least_match(initial_data, adjusted_data, target_indices):
  opponent_index = calc_opponent_index(len(adjusted_data))
  opponent_user_id = adjusted_data[opponent_index]['user_id']
  adjusted_index = -1
  adjusted_match_value = TEMPORARY_INITIAL_MATCH_VALUE
  match_log = []
  for current_index in target_indices:
    current_user_id = initial_data[current_index]['user_id']
    least_num_entrants = CONDITIONAL_LEAST_NUM_ENTRANTS if current_index <= APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM and opponent_index <= APPLY_CONDITIONAL_LEAST_NUM_ENTRANTS_SEED_NUM else 0
    current_match_value = calc_match_point(current_user_id, opponent_user_id, initial_data[current_index].get('hidden_value', 0), initial_data[opponent_index].get('hidden_value', 0), least_num_entrants)

    player_name_for_log = initial_data[current_index].get('player_name', initial_data[current_index].get('gamer_tag', 'Unknown'))
    match_log.extend([current_index, initial_data[current_index]['user_id'], player_name_for_log, current_match_value])
    if (adjusted_match_value <= TEMPORARY_INITIAL_MATCH_VALUE):
      adjusted_index = current_index
      adjusted_match_value = current_match_value
      continue
    if (current_match_value < adjusted_match_value):
      adjusted_index = current_index
      adjusted_match_value = current_match_value
      continue
  return { 'adjusted_index': adjusted_index, 'opponent_index': opponent_index, 'match_log': match_log }

def get_adjusted_result(initial_data):
  adjusted_data = [initial_data[0]]
  match_logs = [[]]
  wave_violations = []

  for i in range(1, len(initial_data)):
    print('target_index', i)

    # Wave-valid な候補を取得。全員対象外なら Wave 制約を無視してフォールバック
    target_indices = get_target_indices(initial_data, adjusted_data)
    wave_ignored = False
    if len(target_indices) == 0:
      target_indices = get_target_indices(initial_data, adjusted_data, ignore_wave=True)
      wave_ignored = True

    if len(target_indices) <= 0:
      break
    if (i < FIXED_SEED_NUM):
      if wave_ignored:
        _record_wave_violation(wave_violations, initial_data, adjusted_data, target_indices[0])
      adjusted_data.append(initial_data[target_indices[0]])
      match_logs.append([])
      continue
    if len(target_indices) <= 1:
      if wave_ignored:
        _record_wave_violation(wave_violations, initial_data, adjusted_data, target_indices[0])
      adjusted_data.append(initial_data[target_indices[0]])
      match_logs.append([])
      continue

    # タイトグループ検出: w人がw枠にしか置けない場合を優先（単一強制を含む）
    tight_group = get_tight_group(initial_data, adjusted_data)
    if tight_group:
      target_set = set(target_indices)
      tight_valid_here = [idx for idx in tight_group if idx in target_set]
      effective_indices = tight_valid_here if tight_valid_here else target_indices
    else:
      effective_indices = target_indices

    # 統一ロジック: best_left_player_based（シードが高い順に確認）→ seed_position_based
    placed = False
    for candidate_index in effective_indices:
      is_adjusted_seed_match_log = is_adjusted_seed(initial_data, adjusted_data, candidate_index)
      if is_adjusted_seed_match_log is not None:
        if wave_ignored:
          _record_wave_violation(wave_violations, initial_data, adjusted_data, candidate_index)
        opponent_idx = calc_opponent_index(len(adjusted_data))
        opponent_player_name = initial_data[opponent_idx].get('player_name', initial_data[opponent_idx].get('gamer_tag', 'Unknown'))
        match_logs.append(['best_left_player_based', opponent_player_name, ''] + is_adjusted_seed_match_log)
        adjusted_data.append(initial_data[candidate_index])
        placed = True
        break

    if not placed:
      least_match_result = get_least_match(initial_data, adjusted_data, effective_indices)
      if wave_ignored:
        _record_wave_violation(wave_violations, initial_data, adjusted_data, least_match_result['adjusted_index'])
      opponent_player_name = initial_data[least_match_result['opponent_index']].get('player_name', initial_data[least_match_result['opponent_index']].get('gamer_tag', 'Unknown'))
      match_logs.append(['seed_position_based', opponent_player_name, ''] + least_match_result['match_log'])
      adjusted_data.append(initial_data[least_match_result['adjusted_index']])

  return { 'adjusted_data': adjusted_data, 'match_logs': match_logs, 'wave_violations': wave_violations }


In [ ]:
def extract_spreadsheet_id(url):
    """
    GoogleスプレッドシートのURLからスプレッドシートIDを抽出する関数。

    Args:
        url (str): GoogleスプレッドシートのURL。

    Returns:
        str: 抽出されたスプレッドシートID、または見つからない場合はNone。
    """
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', url)
    if match:
        return match.group(1)
    return None

In [ ]:
##################### 出力処理 ##########################
from google.colab import runtime, auth

auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

import pandas as pd
import re

SPREADSHEET_URL = userdata.get('SPREADSHEET_URL')
SPREAD_SHEET_ID = extract_spreadsheet_id(SPREADSHEET_URL)
ADJUSTER_INPUT_WORKSHEET_NAME = userdata.get('ADJUSTER_INPUT_WORKSHEET_NAME')

# prompt: データ読み込み
ss_id = SPREAD_SHEET_ID
workbook = gc.open_by_key(ss_id)
worksheet = workbook.worksheet(ADJUSTER_INPUT_WORKSHEET_NAME)
initial_data = worksheet.get_all_records()
header = worksheet.row_values(1)

# Wave パターンワークシートの読み込み（未設定の場合はスキップ）
WAVE_PATTERN_WORKSHEET_NAME = userdata.get('WAVE_PATTERN_WORKSHEET_NAME')
if WAVE_PATTERN_WORKSHEET_NAME:
  try:
    wave_pattern_sheet = workbook.worksheet(WAVE_PATTERN_WORKSHEET_NAME)
    wave_pattern_rows = wave_pattern_sheet.get_all_records()
    wave_pattern = {int(row['pattern']): row['wave'] for row in wave_pattern_rows}
    wave_cycle_length = max(wave_pattern.keys()) if wave_pattern else 1
    print(f'Wave パターン読み込み完了: {wave_pattern}')
  except Exception as e:
    print(f'Wave パターン読み込み失敗: {e}')

# 選手 Wave 希望ワークシートの読み込み（未設定の場合はスキップ、失敗時は中止）
PLAYER_WAVE_WORKSHEET_NAME = userdata.get('PLAYER_WAVE_WORKSHEET_NAME')
if PLAYER_WAVE_WORKSHEET_NAME:
  if 'discriminator' not in header:
    raise ValueError(f'メインシート "{ADJUSTER_INPUT_WORKSHEET_NAME}" に discriminator 列がありません')
  player_wave_sheet = workbook.worksheet(PLAYER_WAVE_WORKSHEET_NAME)
  player_wave_rows = player_wave_sheet.get_all_records()
  main_discriminators = {str(row.get('discriminator', '')) for row in initial_data}
  missing = [str(row['discriminator']) for row in player_wave_rows if str(row['discriminator']) not in main_discriminators]
  if missing:
    raise ValueError(f'メインシートに存在しない discriminator があります: {missing}')
  for row in player_wave_rows:
    disc = str(row['discriminator'])
    allowed_waves_map.setdefault(disc, []).append(row['wave'])
  print(f'選手 Wave 希望設定読み込み完了: {len(player_wave_rows)} 件')

# Add original_input_order to each item in initial_data based on its row index
for idx, row in enumerate(initial_data):
    row['original_input_order'] = idx + 1

adjusted_result = get_adjusted_result(initial_data)

# データ加工
for i in range(0, len(adjusted_result['adjusted_data'])):
  current_row_data = adjusted_result['adjusted_data'][i]
  current_row_data['original_phaseseed'] = current_row_data['original_input_order']
  current_row_data['phaseseed'] = i + 1
  current_row_data['adjusted_wave'] = get_wave(i)  # 調整後の位置に対応するWave

output_data_keys = worksheet.row_values(1).copy()
if 'phaseseed' not in output_data_keys:
    try:
        if 'original_phaseseed' in output_data_keys:
            idx = output_data_keys.index('original_phaseseed')
            output_data_keys.insert(idx, 'phaseseed')
        else:
            output_data_keys.append('phaseseed')
    except ValueError:
        output_data_keys.append('phaseseed')

if 'original_phaseseed' not in output_data_keys:
    output_data_keys.append('original_phaseseed')

# originalPhaseseedの右にwaveを挿入
if 'adjusted_wave' not in output_data_keys:
    idx = output_data_keys.index('original_phaseseed')
    output_data_keys.insert(idx + 1, 'adjusted_wave')

# prompt: データ保存
adjusted_data = adjusted_result['adjusted_data']
match_logs = adjusted_result['match_logs']
wave_violations = adjusted_result.get('wave_violations', [])

new_header_for_display = output_data_keys.copy()
new_header_for_display.append("")
new_header_for_display.append("match_type")
new_header_for_display.append("projected_opponent")
new_header_for_display.append("")
new_header_for_display.append("note")

for i, col in enumerate(new_header_for_display):
    if col == "phaseseed":
        new_header_for_display[i] = "phaseseed"
    elif col == "original_phaseseed":
        new_header_for_display[i] = "originalPhaseseed"
    elif col == "adjusted_wave":
        new_header_for_display[i] = "wave"

matrix_data = [new_header_for_display]
for i in range(0, len(adjusted_data)):
    row = adjusted_data[i]
    row_data = []
    for column_key in output_data_keys:
      if column_key in row:
        row_data.append(row[column_key])
      else:
        row_data.append(None)
    row_data.append("")
    row_data.extend(match_logs[i])
    matrix_data.append(row_data)

# Wave 制約違反があった場合は末尾に警告セクションを追加
if wave_violations:
  matrix_data.append([])
  matrix_data.append(['[警告] Wave希望を満たせなかった選手'])
  matrix_data.append(['phaseseed', 'player_name', 'wave', 'allowed_waves'])
  for v in wave_violations:
    matrix_data.append([v['phaseseed'], v['player_name'], v['wave'], ','.join(v['allowed_waves'])])

output_data = list(map(lambda row: list(map(lambda cell: str(cell), row)), matrix_data))
adjusted_worksheet = workbook.add_worksheet("adjusted_" + get_yyyymmddhhmmss(), len(output_data) + 5, 1000)
adjusted_worksheet.append_rows(output_data)

##################### ここまで ############################################

# 片付け処理
clean_up_repository(repository_path)


In [ ]:
#### 値のチェックだけ

print(calc_match_point(134839, 2028400, 0, 0, CONDITIONAL_LEAST_NUM_ENTRANTS))

0
